**[Source]** Donghwan Project (디코딩 비교: Greedy vs Beam) + Jisoo Project 13단계(전체 Validation Greedy/Beam3 예측)
**[Status]** ADAPTED
**[Role]** Greedy를 기준선으로 Beam Search(num_beams=3)를 같은 checkpoint·같은 Validation 전체에서 비교하고, 07번 사전 등록 규칙으로 채택 여부를 판정
**[Modification]** 지수가 이미 생성한 전체 Validation 예측을 재사용(07번에서 id·input·target 일치 확인). 동환식 지표(Balanced EM 등)를 추가 계산. 추가 sweep(beam 수·length/repetition penalty)은 GPU가 필요해 기본 꺼짐.
**Test는 열지 않는다. 디코딩 설정은 Validation으로만 선택한다.**

# 10. 디코딩 비교 — Greedy vs Beam Search(3)
- **Greedy**: 매 step 확률이 가장 높은 토큰 1개를 선택. 빠르지만 한 번 잘못 고르면 되돌릴 수 없다.
- **Beam Search(`num_beams=k`)**: 매 step 누적 로그확률 상위 k개 후보 시퀀스를 유지. 전체 시퀀스 확률이 높은 문장을 찾지만 계산량이 늘고, **확률이 높은 문장 ≠ 사람이 보기에 정확한 교정**일 수 있다(빈 출력·반복·과도하게 긴 출력이 늘 수 있음). beam이 크다고 항상 좋아지는 것은 아니다.
- `length_penalty`(beam 점수의 길이 보정), `repetition_penalty`(이미 나온 토큰 확률 감쇠), `early_stopping`(모든 beam이 끝나면 조기 종료)은 beam 계열에서 의미가 있으며 이 노트북에서는 **기본값(1.0/1.0)으로 고정**했다. 바꾼 실험은 아래 (선택) GPU sweep에만 있다.
- 샘플링(Top-k/Top-p/Temperature)은 **사용하지 않는다**: 교정은 정답이 사실상 하나인 과제이고, 실행마다 결과가 달라져 재현·평가가 어렵기 때문이다(이 판단은 이론적 근거이며 이번에 실험으로 검증하지는 않았다).

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 전체 Validation 예측 로드 + 같은 행인지 검증
VAL = common.read_split(P, "validation", columns=["document_id", "utterance_id", "input", "target"])
SRC, TGT, DOC = [r["input"] for r in VAL], [r["target"] for r in VAL], [r["document_id"] for r in VAL]
DEC = P.J_OUT / "decoding_comparison"; PRED = {}
for name, f in (("Greedy", "val_predictions_greedy.jsonl"), ("Beam3", "val_predictions_beam3.jsonl")):
    rows = common.read_jsonl(DEC / f)
    assert [r["utterance_id"] for r in rows] == [r["utterance_id"] for r in VAL] and [r["input"] for r in rows] == SRC and [r["target"] for r in rows] == TGT, f"{name}: Validation과 불일치"
    PRED[name] = [str(r["prediction"]) for r in rows]; print(f"{name}: {len(rows):,}행 검증 통과 | 빈 출력 {sum(1 for x in PRED[name] if not x.strip())}")
DJ = json.loads((DEC / "decoding_comparison_results.json").read_text(encoding="utf-8"))
print("생성 설정 Greedy:", DJ["greedy_kwargs"]); print("생성 설정 Beam3 :", DJ["beam_kwargs"])

Greedy: 54,730행 검증 통과 | 빈 출력 0
Beam3: 54,730행 검증 통과 | 빈 출력 0
생성 설정 Greedy: {'do_sample': False, 'max_new_tokens': 72, 'length_penalty': 1.0, 'repetition_penalty': 1.0, 'no_repeat_ngram_size': 0, 'num_beams': 1}
생성 설정 Beam3 : {'do_sample': False, 'max_new_tokens': 72, 'length_penalty': 1.0, 'repetition_penalty': 1.0, 'no_repeat_ngram_size': 0, 'num_beams': 3}


In [3]:
# [셀 2] 지표(동환식 + 지수식) — 전체 Validation
t0 = time.time(); need = np.array([s != t for s, t in zip(SRC, TGT)])
ROW = {m: km.score_rows(PRED[m], TGT) for m in PRED}
MET = {(m, nf): km.generation_metrics(SRC, TGT, PRED[m], nfkc=(nf == "NFKC"), with_chrf=(nf == "NFKC")) for m in PRED for nf in ("NFKC", "raw")}
for m in PRED:
    r2 = float(ROW[m]["R2_어절"].mean()); rec = DJ["table"]["Greedy" if m == "Greedy" else "Beam Search(3)"]["ROUGE-2(어절)"]
    assert abs(r2 - rec) < 1e-3, f"{m}: 지수 R2 재현 실패 {r2} vs {rec}"
print("지수 ROUGE-2 재현 확인(오차<1e-3) | 계산 시간 %.0f초" % (time.time() - t0))
cols = ["exact_match", "need_correction_em", "unchanged_em", "balanced_em", "cer", "chrf", "rouge2_donghwan", "over_correction_rate", "miss_rate"]
df = pd.DataFrame({f"{m}|{nf}": v for (m, nf), v in MET.items()}).T[cols]; print(df.round(4).to_string())
for m in PRED: print(m, "어절 R1/R2/RL:", round(float(ROW[m]["R1_어절"].mean()), 4), round(float(ROW[m]["R2_어절"].mean()), 4), round(float(ROW[m]["RL_어절"].mean()), 4), "| 문자 R2:", round(float(ROW[m]["R2_문자"].mean()), 4))

지수 ROUGE-2 재현 확인(오차<1e-3) | 계산 시간 14초
             exact_match  need_correction_em  unchanged_em  balanced_em     cer    chrf  rouge2_donghwan  over_correction_rate  miss_rate
Greedy|NFKC       0.6965              0.6693        0.8707       0.7700  0.0355  0.9389           0.8080                0.1293     0.0289
Greedy|raw        0.5269              0.5129        0.6169       0.5649  0.1025     NaN           0.7073                0.3831     0.0275
Beam3|NFKC        0.6978              0.6725        0.8595       0.7660  0.0353  0.9397           0.8090                0.1405     0.0262
Beam3|raw         0.5276              0.5156        0.6047       0.5601  0.1022     NaN           0.7080                0.3953     0.0247
Greedy 어절 R1/R2/RL: 0.7821 0.7823 0.7821 | 문자 R2: 0.8603
Beam3 어절 R1/R2/RL: 0.7827 0.7829 0.7826 | 문자 R2: 0.8608


In [4]:
# [셀 3] Beam3 − Greedy 차이(문서 단위 bootstrap) + 사전 등록 규칙 5개
import re
docs = np.array(DOC); ci = lambda a, b, mask=None: km.boot_paired_diff(a if mask is None else a[mask], b if mask is None else b[mask], docs if mask is None else docs[mask])
d_all = ci(ROW["Beam3"]["R2_어절"], ROW["Greedy"]["R2_어절"]); d_need = ci(ROW["Beam3"]["R2_어절"], ROW["Greedy"]["R2_어절"], need)
ex = {m: np.array([km.normalize_for_evaluation(p) == km.normalize_for_evaluation(t) for p, t in zip(PRED[m], TGT)]) for m in PRED}
d_bal = km.boot_balanced_em_diff(ex["Beam3"], ex["Greedy"], need, DOC)
T = pd.DataFrame([{"비교(Beam3−Greedy)": k, "차이": round(v[0], 4), "CI95 하한": round(v[1], 4), "CI95 상한": round(v[2], 4)} for k, v in {"전체 어절 R2": d_all, "교정 필요 행 어절 R2": d_need, "Balanced EM (NFKC)": d_bal}.items()]); print(T.to_string(index=False))
rep = re.compile(r"(.{2,}?)\1{3,}")
def n_rep(m): return sum(1 for p, t in zip(PRED[m], TGT) if rep.search(p) and not rep.search(t))
def n_empty(m): return sum(1 for p in PRED[m] if not p.strip())
oc = {m: MET[(m, "NFKC")]["over_correction_rate"] for m in PRED}
ratio_same_session = DJ["same_session_1000rows_sec"]["ratio"]; ratio_total = DJ["beam_total_time_sec"] / DJ["greedy_total_time_sec"]
crit = {"1. 전체 R2 차이 CI 하한 > 0": d_all[1] > 0, "2. 교정 필요 행 R2 차이 CI 하한 > 0": d_need[1] > 0,
        "3. 과교정률(NFKC, 원문 유지 행) 증가 ≤ 0.5%p": (oc["Beam3"] - oc["Greedy"]) * 100 <= 0.5,
        "4. 빈 출력·반복 생성 후보가 늘지 않음": n_empty("Beam3") <= n_empty("Greedy") and n_rep("Beam3") <= n_rep("Greedy"), "5. 시간 비율 ≤ 5배(같은 세션 1,000행)": ratio_same_session <= 5}
print("\n과교정률(NFKC): Greedy %.4f / Beam3 %.4f (차이 %+.2f%%p)" % (oc["Greedy"], oc["Beam3"], (oc["Beam3"] - oc["Greedy"]) * 100))
print("빈 출력: Greedy %d / Beam3 %d | 반복 생성 후보(내 정규식 정의): Greedy %d / Beam3 %d" % (n_empty("Greedy"), n_empty("Beam3"), n_rep("Greedy"), n_rep("Beam3")))
print("시간: 같은 세션 1,000행 비율 %.2f배 | 전체 실행 벽시계 비율 %.1f배 (Greedy %.1f분, Beam3 %.1f분 — 배치 크기 등 조건이 달라 직접 비교는 부정확)" % (ratio_same_session, ratio_total, DJ["greedy_total_time_sec"] / 60, DJ["beam_total_time_sec"] / 60))
print(pd.Series(crit).to_string()); use_beam = all(crit.values()); SELECTED_DECODING = "beam3" if use_beam else "greedy"
print("\n결론: 5개 규칙 모두 충족 시에만 Beam 채택 →", SELECTED_DECODING, "| 지수 13단계 선택과", "일치" if SELECTED_DECODING == DJ["selected"] else "불일치")
(P.RUNS / "decoding_comparison_10.json").write_text(json.dumps({"selected_decoding": SELECTED_DECODING, "criteria": {k: bool(v) for k, v in crit.items()}, "diffs": {"all_r2": d_all, "need_r2": d_need, "balanced_em": d_bal},
    "metrics": {f"{m}|{nf}": v for (m, nf), v in MET.items()}, "time": {"same_session_ratio": ratio_same_session, "total_ratio": ratio_total}, "scope": "전체 Validation, Test 미사용, 샘플링 미사용"}, ensure_ascii=False, indent=2, default=float), encoding="utf-8")

  비교(Beam3−Greedy)    차이  CI95 하한  CI95 상한
        전체 어절 R2  0.0006    -0.0000     0.0012
교정 필요 행 어절 R2  0.0012     0.0006     0.0019
  Balanced EM (NFKC) -0.0041    -0.0057    -0.0026

과교정률(NFKC): Greedy 0.1293 / Beam3 0.1405 (차이 +1.12%p)
빈 출력: Greedy 0 / Beam3 0 | 반복 생성 후보(내 정규식 정의): Greedy 9 / Beam3 9
시간: 같은 세션 1,000행 비율 2.59배 | 전체 실행 벽시계 비율 8.9배 (Greedy 9.2분, Beam3 82.1분 — 배치 크기 등 조건이 달라 직접 비교는 부정확)
1. 전체 R2 차이 CI 하한 > 0                     False
2. 교정 필요 행 R2 차이 CI 하한 > 0              True
3. 과교정률(NFKC, 원문 유지 행) 증가 ≤ 0.5%p    False
4. 빈 출력·반복 생성 후보가 늘지 않음            True
5. 시간 비율 ≤ 5배(같은 세션 1,000행)            True

결론: 5개 규칙 모두 충족 시에만 Beam 채택 → greedy | 지수 13단계 선택과 일치


2627

In [5]:
# [셀 4] Greedy와 Beam3의 출력이 다른 행 — 실제 사례(Validation)
diff_idx = [i for i in range(len(VAL)) if PRED["Greedy"][i] != PRED["Beam3"][i]]
print(f"출력이 다른 행: {len(diff_idx):,} / {len(VAL):,} ({100*len(diff_idx)/len(VAL):.2f}%)")
b_better = [i for i in diff_idx if ROW["Beam3"]["R2_어절"][i] > ROW["Greedy"]["R2_어절"][i]]; g_better = [i for i in diff_idx if ROW["Greedy"]["R2_어절"][i] > ROW["Beam3"]["R2_어절"][i]]
print("Beam3가 더 높은 행:", len(b_better), "| Greedy가 더 높은 행:", len(g_better), "| 같은 점수:", len(diff_idx) - len(b_better) - len(g_better))
rng = np.random.default_rng(42)
for title, pool in (("Beam3가 더 좋은 사례", b_better), ("Greedy가 더 좋은 사례", g_better)):
    print("\n[", title, "]")
    for i in rng.choice(pool, size=min(4, len(pool)), replace=False):
        print(f" 입력 : {SRC[i]}\n 정답 : {TGT[i]}\n Greedy: {PRED['Greedy'][i]}\n Beam3 : {PRED['Beam3'][i]}\n")

출력이 다른 행: 1,439 / 54,730 (2.63%)
Beam3가 더 높은 행: 480 | Greedy가 더 높은 행: 363 | 같은 점수: 596

[ Beam3가 더 좋은 사례 ]
 입력 : 헐 죽기까지
 정답 : 헐, 죽기까지.
 Greedy: 헐, 죽기까지
 Beam3 : 헐, 죽기까지.

 입력 : 어떤 사람 3개월 하루빼고 매일 한 인증글 올라와신디 된댘ㅋㅋㅋㅋㅋㅋ
 정답 : 어떤 사람 3개월 하루 빼고 매일 한 인증 글 올라왔는데 된대. ㅋㅋㅋㅋㅋㅋㅋ
 Greedy: 어떤 사람 3개월 하루 빼고 매일 한 인증 글 올라와 있던데 된대. ᄏᄏᄏᄏᄏᄏᄏ
 Beam3 : 어떤 사람 3개월 하루 빼고 매일 한 인증 글 올라와서 된대. ᄏᄏᄏᄏᄏᄏᄏ

 입력 : 헉 와사비간장은 찍어먹을 생각 해본 적 없는데 특이하다
 정답 : 헉, 와사비 간장은 찍어 먹을 생각해 본 적 없는데 특이하다.
 Greedy: 헉, 와사비간장은 찍어 먹을 생각 해 본 적 없는데 특이하다.
 Beam3 : 헉, 와사비간장은 찍어 먹을 생각해 본 적 없는데 특이하다.

 입력 : 아는분이 요리고수신가봐요
 정답 : 아는 분이 여리고 수신가 봐요
 Greedy: 아는 분이 요리 고수인가 봐요.
 Beam3 : 아는 분이 요리고수신가 봐요.


[ Greedy가 더 좋은 사례 ]
 입력 : 네! 얼리어답터는 아니지만 꼭 써봐야 직성이 풀리는 편이에요 ㅎㅎ
 정답 : 네! 얼리 어댑터는 아니지만 꼭 써 봐야 직성이 풀리는 편이에요. ㅎㅎ
 Greedy: 네! 얼리 어답터는 아니지만 꼭 써봐야 직성이 풀리는 편이에요. ᄒᄒ
 Beam3 : 네! 얼리어답터는 아니지만 꼭 써봐야 직성이 풀리는 편이에요. ᄒᄒ

 입력 : 내 생각에는 그냥 처음에 아예 택배 받기 힘들다고 말을 해서 그냥 잘못 보낸 것 같은데
 정답 : 내 생각에는 그냥 처음에 아예 택배 받기 힘들다고 말을 해서 그냥 잘못 보낸 것 같은데
 Greedy: 내 생각에는 그냥 처음에 아예 택배 받기 힘들다고 말을 해서 그냥 잘

## (선택) GPU sweep — beam 수 · length_penalty · repetition_penalty
아래 셀은 `RUN_SWEEP=True`이고 GPU·torch·체크포인트가 있을 때만 실행된다. **Validation 앞 2,000행**에서 한 번에 한 조건만 바꿔 비교한다. 조합이 8개 이상이므로 **탐색용**이며, 어떤 설정이 좋아 보여도 (1) 문서 단위 CI 하한 > 0 이고 (2) 전체 Validation에서 다시 확인되기 전에는 채택하지 않는다(여러 설정 중 최고만 고르면 우연히 높은 값을 고르게 됨). 작성 환경에서 실행해 보지 못했다 — 실행 후 확인.

In [6]:
# [셀 5] (선택) sweep — 기본 꺼짐
RUN_SWEEP = False
SWEEP_STATUS = {"run": False}
try:
    import torch; HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
CK = P.J_OUT / "checkpoints" / "pkot5_full_1epoch"
if not RUN_SWEEP or not HAS_GPU or not CK.exists():
    print("sweep 미실행: RUN_SWEEP=%s, GPU=%s, checkpoint=%s" % (RUN_SWEEP, HAS_GPU, CK.exists()))
else:
    import seq2seq_tools as S2S
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(str(CK)); model = AutoModelForSeq2SeqLM.from_pretrained(str(CK)).cuda().eval()
    N = 2000; cfgs = [{"name": "greedy", "num_beams": 1}, {"name": "beam2", "num_beams": 2}, {"name": "beam3", "num_beams": 3}, {"name": "beam5", "num_beams": 5},
                      {"name": "beam3_lp0.8", "num_beams": 3, "length_penalty": 0.8}, {"name": "beam3_lp1.2", "num_beams": 3, "length_penalty": 1.2},
                      {"name": "greedy_rp1.2", "num_beams": 1, "repetition_penalty": 1.2}, {"name": "beam3_rp1.2", "num_beams": 3, "repetition_penalty": 1.2}]
    out, preds = [], {}
    for c in cfgs:
        kw = {k: v for k, v in c.items() if k != "name"}; kw.update(do_sample=False); t0 = time.time()
        preds[c["name"]] = S2S.generate(model, tok, SRC[:N], 72, 72, kw, batch_size=16); sec = time.time() - t0
        m = km.generation_metrics(SRC[:N], TGT[:N], preds[c["name"]]); m.update(name=c["name"], sec=sec, r2_word=float(np.mean(km.score_rows(preds[c["name"]], TGT[:N])["R2_어절"]))); out.append(m)
    SW = pd.DataFrame(out).set_index("name"); print(SW[["r2_word", "balanced_em", "cer", "over_correction_rate", "miss_rate", "sec"]].round(4).to_string())
    SW.to_csv(P.RUNS / "decoding_sweep_val2000.csv", encoding="utf-8-sig"); SWEEP_STATUS = {"run": True, "n": N}

sweep 미실행: RUN_SWEEP=False, GPU=False, checkpoint=True


## 해석
- **지수 ROUGE-2 재현 확인**(오차 <1e-3) 후 계산했다. 전체 Validation 54,730행, 같은 체크포인트·같은 최대 생성 길이(72)·기본 length/repetition penalty.
- **Beam3 − Greedy**: 전체 어절 R2 +0.0006(CI95 −0.0000~0.0012, 0을 사실상 포함), 교정 필요 행 R2 +0.0012(0.0006~0.0019, 0 제외). 즉 **점수 차이는 소수 셋째 자리 수준**이다. 반면 **Balanced EM(NFKC)은 Beam3가 −0.0041(−0.0057~−0.0026)로 오히려 낮다.** 이유는 원문 유지 행 EM이 0.871→0.860로 떨어졌기 때문이다(과교정률 12.9%→14.1%, +1.12%p). 원문이 이미 맞는 문장을 Beam3가 더 자주 바꾼다는 뜻이며, 그 이유(예: 문장 끝 부호 추가 경향)는 이번에 분석하지 않았다.
- **사전 등록 규칙 5개 중 충족 3개(2·4·5), 미충족 2개(1·3) → Greedy 유지.** 지수 13단계의 선택과 일치한다. 결론이 근소한 R2 이득을 얻기 위해 과교정을 늘리는 쪽이므로 Beam을 채택하지 않는 것이 규칙과 데이터 양쪽에 부합한다.
- **비용**: 같은 세션 1,000행 기준 Beam3는 Greedy의 2.59배(규칙 5 충족). 그러나 전체 실행 벽시계는 82.1분 vs 9.2분(8.9배)으로 기록되어 있다. 두 실행의 배치 크기 등이 달라 직접 비교는 부정확하지만, 실제 대규모 추론에서는 Beam3가 비용이 더 크다는 방향은 분명하다. 이득이 거의 없는 상황에서 비용만 큰 선택이다.
- **출력이 다른 행은 1,439행(2.63%)뿐**이며 Beam3가 더 높은 행 480, Greedy가 더 높은 행 363, 같은 점수 596이다. 즉 디코딩 설정이 결과에 미치는 영향 자체가 작다. 이 모델의 오류를 줄이려면 디코딩보다 후처리(12번)·라벨·데이터 쪽이 훨씬 큰 지렛대라는 것이 데이터로 확인된다.
- **표시상 주의**: 사례 출력의 `ᄏᄏᄏ`는 원문의 `ㅋㅋㅋ`가 tokenizer NFKC 정규화로 바뀐 것이다(12번에서 원인과 후처리 실험). 평가 지표의 NFKC 값에서는 이 차이가 가려진다.
- **미실행/한계**: 추가 sweep(beam 2·5, length_penalty, repetition_penalty)은 GPU가 없어 **실행하지 않았다**(RUN_SWEEP=False). 따라서 “beam 수를 늘리면 어떻게 되는지”, “penalty 조정이 효과가 있는지”는 이번 자료로 주장할 수 없다. 샘플링(Top-k/Top-p/Temperature)은 사용하지 않았고, 사용하지 않은 이유는 이론적 판단이며 실험으로 검증한 것이 아니다.